# 01 · The frames — a typed storage contract for `(time, unit)` arrays

**What this teaches:** what a `views_frames` frame *is* and what it can do as a
storage device — the immutable, numpy-only **array + identifier** value objects at
the root of the VIEWS dependency graph, and why they exist (to replace the
list-in-cell `object`-dtype DataFrame that caused the ~33× memory blow-up /
report-stage OOM, README §7).

**Audience:** a consumer-repo author about to migrate a DataFrame-based path onto frames.

> **How to read this notebook.** Everything below uses only the **public, frozen**
> `views_frames` API, on **synthetic data** generated in-repo with fixed seeds
> (`notebooks/_synthetic.py`). It is an un-gated dev artifact — the package never
> imports it. A full *Run All* finishes in well under a minute.

### Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# `_synthetic.py` sits next to this notebook; make it importable whether the kernel
# starts in notebooks/ or at the repo root.
for _p in (Path.cwd(), Path.cwd() / "notebooks"):
    if (_p / "_synthetic.py").exists():
        sys.path.insert(0, str(_p))
        break
import _synthetic as syn

from views_frames import (
    FeatureFrame,
    FrameMetadata,
    PredictionFrame,
    SpatialLevel,
    SpatioTemporalIndex,
    TargetFrame,
)

RNG_SEED = 0
np.set_printoptions(precision=3, suppress=True)

## 1 · Motivation — the problem the frame solves

A forecast posterior is **`S` samples per `(month, cell)` row**. The pre-frames
encoding stored that as a *list-in-cell* `DataFrame` — one column whose every cell
holds a Python `list` of `S` floats. That is an `object`-dtype column of pointers to
millions of boxed Python `float`s: it cannot be vectorised, it cannot memory-map, and
it blows up in RAM (the report-stage OOM, README §7).

The frame replaces it with a single **dense `float32 (N, S)`** array plus **integer
identifiers**. Below: the same posterior both ways, and the memory ratio.

In [ ]:
N, S = 2_000, 256

# dense, frame-style: one contiguous float32 (N, S) block
dense = np.random.default_rng(RNG_SEED).standard_normal((N, S)).astype(np.float32)

# list-in-cell, DataFrame-style: an object array of Python lists of Python floats
list_in_cell = np.empty(N, dtype=object)
for i in range(N):
    list_in_cell[i] = [float(x) for x in dense[i]]

def list_in_cell_bytes(col, sample_rows=200):
    """Rough memory of the object column: array of pointers + lists + boxed floats."""
    rows = col[:sample_rows]
    per_row = np.mean([
        sys.getsizeof(lst) + sum(sys.getsizeof(x) for x in lst) for lst in rows
    ])
    return col.nbytes + per_row * len(col)

dense_mb = dense.nbytes / 1e6
listcell_mb = list_in_cell_bytes(list_in_cell) / 1e6
print(f"dense  float32 (N, S):  {dense_mb:7.2f} MB")
print(f"list-in-cell (object):  {listcell_mb:7.2f} MB")
print(f"blow-up factor:         {listcell_mb / dense_mb:7.1f}x")
print("\n(README §7 reports ~33x on the production payload — bigger lists, "
      "wider rows. The contract exists to make that encoding unrepresentable.)")

## 2 · `SpatioTemporalIndex` — the shared identifier object

Every frame carries a `SpatioTemporalIndex`: integer `time` + `unit` arrays and a
`SpatialLevel` (`cm` country-month / `pgm` PRIO-GRID-month — a label-only vocabulary,
never geography). The index is the **one genuinely-reused primitive**: it provides
pure-numpy, same-level alignment (`searchsorted` / `intersect` / `reindex`), the
numpy unwrap of `pd.Index.get_indexer`.

In [ ]:
time = np.full(5, 500, dtype=np.int64)              # month_id 500
unit = np.array([1001, 1002, 1003, 1004, 1005], np.int64)  # priogrid_id
index = SpatioTemporalIndex(time=time, unit=unit, level=SpatialLevel.PGM)

print("level        :", index.level, "->", index.level.index_names)
print("n_rows       :", index.n_rows)
print("identifiers  :", {k: v.tolist() for k, v in index.identifiers.items()})

Same-level alignment is the operation consumers actually reuse — *where* does each
row of one index land in another, with no pandas:

In [ ]:
subset = SpatioTemporalIndex(
    time=np.array([500, 500], np.int64),
    unit=np.array([1003, 1001], np.int64),
    level=SpatialLevel.PGM,
)
print("positions of `subset` rows within `index`:", index.searchsorted(subset))
print("is `index` a superset of `subset`?       :", index.is_superset_of(subset))

## 3 · The three frames as *siblings* (no base class — ADR-011)

There is **no `_BaseFrame`**. `PredictionFrame`, `TargetFrame` and `FeatureFrame` are
independent sibling classes that *share an index*, not an inheritance tree — so a
change to one cannot silently reshape the others (ADR-011, Option C). The three differ
only in the shape of their value block:

| frame | values | meaning |
|---|---|---|
| `PredictionFrame` | `(N, S)` | a posterior of `S` samples per row |
| `TargetFrame` | `(N, 1)` | an observed actual (an explicit 1-sample frame, ADR-012) |
| `FeatureFrame` | `(N, F, S)` | `F` model-input channels per row |

`cm`/`pgm` is a **`SpatialLevel` value on the index**, never a class axis. Below we
build all three on one shared index from the synthetic zoo (one posterior per
distribution shape).

In [ ]:
zoo = syn.zoo_frames(n_samples=256, seed=RNG_SEED)
pred, tgt, feat = zoo.prediction, zoo.target, zoo.features

for name, frame in [("PredictionFrame", pred), ("TargetFrame", tgt), ("FeatureFrame", feat)]:
    print(f"{name:16s} values{str(frame.values.shape):12s} "
          f"sample_count={frame.sample_count:<4d} is_sample={frame.is_sample}")

print("\nall three share ONE index:",
      pred.index == tgt.index == feat.index)
print("shapes on the index   :", zoo.names)

`is_sample` (from the `Sampled` protocol) is the honest distinction — a
`PredictionFrame` of `S>1` draws is a sample; a `TargetFrame` is a single realised
value (`is_sample == False`), not "a posterior with one draw". A quick look at what a
"posterior per row" means:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 2.6))
for ax, i in zip(axes, [0, 1, 5]):           # zero-inflated, skewed, bimodal
    ax.hist(pred.values[i], bins=40, color="steelblue")
    ax.axvline(tgt.values[i, 0], color="crimson", lw=2, label="actual")
    ax.set_title(zoo.names[i], fontsize=9)
    ax.set_yticks([])
axes[0].legend(fontsize=8)
fig.suptitle("one PredictionFrame row = one posterior; the TargetFrame row = its actual", fontsize=10)
plt.tight_layout(); plt.show()

## 4 · Immutability & structural sharing (register C-07)

Frames are **value objects**: every operation returns a *new* frame and never mutates
the input. Structural operations (`with_metadata`, slicing into a shared buffer) are
**zero-copy** — they share the underlying values array; only operations that must
allocate (a row `select`, a reduction) copy. This is the contract that lets the leaf
be passed across repos without defensive copying.

In [ ]:
original_values = pred.values.copy()      # snapshot to prove non-mutation

# with_metadata: a NEW frame that SHARES the values buffer (zero-copy)
tagged = pred.with_metadata(FrameMetadata(run_id="demo-run", data_version="v2026.06"))
print("with_metadata shares the values buffer :", np.shares_memory(pred.values, tagged.values))
print("new metadata on the new frame          :", tagged.metadata.run_id, tagged.metadata.data_version)

# select: a NEW frame that COPIES (row gather cannot be a view)
first_three = pred.select(np.array([0, 1, 2]))
print("select copies (no shared buffer)       :", not np.shares_memory(pred.values, first_three.values))
print("select result shape                    :", first_three.values.shape)

# a reduction allocates a new (N, 1) block; the source is untouched
row_means = pred.values.mean(axis=1, keepdims=True)
print("source frame never mutated             :", np.array_equal(pred.values, original_values))

**Why it matters:** because `with_metadata` shares the buffer, tagging provenance
onto a multi-gigabyte posterior is free; because `select` copies, a consumer can hand
out a slice without fear the parent changes under it. The frame chooses copy-vs-share
by what is *safe*, not by guesswork — see §8 for the provenance header `with_metadata`
attaches.

## 5 · Cross-level alignment (ADR-014)

Grid (`pgm`) rows often need rolling up to their country (`cm`) — but a cell's country
**changes by month**, so the join is keyed by `(time, unit)`, not `unit` alone. The
leaf owns the *operation* (`cross_level_align`); the **mapping is injected by the
caller** and is never embedded or fetched here. This is the rule that keeps geography
out of the leaf (ADR-014): the frame knows *how* to remap, never *what the map is*.

In [ ]:
sc = syn.cm_pgm_scenario(n_samples=8, seed=RNG_SEED, lattice=(2, 2), n_months=1)
pgm_index = sc.pgm_index
print("pgm rows (time, unit):",
      list(zip(pgm_index.time.tolist(), pgm_index.unit.tolist())))
print("injected (time, unit) -> country mapping:", sc.mapping)

# remap every pgm row's unit to its country, via the INJECTED mapping; time is kept
cm_index = pgm_index.cross_level_align(sc.mapping, SpatialLevel.CM)
print("\nafter cross_level_align -> level:", cm_index.level)
print("remapped units (country_id)    :", cm_index.unit.tolist())
print("time preserved row-for-row     :", np.array_equal(cm_index.time, pgm_index.time))

The leaf never guesses a mapping — an uncovered `(time, unit)` **fails loud**, and a
grid-scale mapping can be injected as columnar arrays (`cross_level_align_arrays`) to
skip building a 10M-key dict:

In [ ]:
# columnar form (what a producer holding the mapping as arrays would use)
cm_arrays = pgm_index.cross_level_align_arrays(sc.map_keys, sc.map_vals, SpatialLevel.CM)
print("array form agrees with dict form:", np.array_equal(cm_arrays.unit, cm_index.unit))

# a mapping that omits a row is rejected — the leaf does not invent geography
try:
    pgm_index.cross_level_align({(500, 1000): 70}, SpatialLevel.CM)  # missing 3 rows
except ValueError as e:
    print("fails loud on an incomplete mapping:", str(e)[:60], "...")

## 6 · Serialization round-trips

Two codecs, both proving `frame in == frame out`:

* **npz** — the native format. `frame.save(dir)` / `Frame.load(dir)`, mmap-capable for
  large posteriors. This is the method on the frame.
* **arrow/parquet** — the scalable flat-columnar interchange format (one scalar cell per
  `(time, unit, sample)`), under `views_frames.io.arrow`. It is the **only** place
  `pyarrow` may be imported.

In [ ]:
import tempfile

# native npz round-trip on the frame itself
with tempfile.TemporaryDirectory() as d:
    pred.save(d)
    reloaded = PredictionFrame.load(d)
print("npz round-trip preserves values :", np.array_equal(reloaded.values, pred.values))
print("npz round-trip preserves index  :", reloaded.index == pred.index)

**The `to_parquet` ergonomic gap (register D-11).** There is deliberately *no*
`frame.to_parquet()` method. To use the parquet codec you reach into `io.arrow` and
**destructure the frame's state** yourself — `arrow.save` takes the raw arrays, not a
frame. That friction is a recorded *demand signal* (D-11), not a reason to bolt a
serialization method onto the frozen value object:

In [ ]:
from views_frames.io import arrow  # the one module allowed to import pyarrow

with tempfile.TemporaryDirectory() as d:
    parquet_path = Path(d) / "pred.parquet"
    # no frame.to_parquet — destructure the frame's state into the codec (D-11)
    arrow.save(
        parquet_path,
        values=pred.values,
        time=pred.index.time,
        unit=pred.index.unit,
        level=pred.index.level.value,
        metadata=pred.metadata.to_dict(),
    )
    state = arrow.load(parquet_path)            # -> a state dict, not a frame
    rebuilt = PredictionFrame(
        state["values"],
        SpatioTemporalIndex(state["time"], state["unit"], SpatialLevel(state["level"])),
        FrameMetadata.from_dict(state["metadata"]),
    )
print("parquet round-trip preserves values:", np.array_equal(rebuilt.values, pred.values))
print("parquet round-trip preserves index :", rebuilt.index == pred.index)

## 7 · Fail-loud construction (ADR-008/009)

The contract is enforced **at construction** — a malformed frame cannot exist. The
guarantee is *structural* (dtype / shape / completeness), not temporal: `time` is an
opaque integer, so the leaf never judges its range or monotonicity (register C-11). A
downstream repo can therefore trust the shape of any frame it is handed without
re-validating. Each attempt below raises at build time:

In [ ]:
good_index = SpatioTemporalIndex(
    time=np.array([500, 500], np.int64),
    unit=np.array([1, 2], np.int64),
    level=SpatialLevel.PGM,
)

def expect_raise(label, thunk):
    try:
        thunk()
    except (ValueError, TypeError) as e:
        print(f"{label:34s} -> {type(e).__name__}: {str(e)[:54]}")
    else:
        print(f"{label:34s} -> NO ERROR (contract hole!)")

# object dtype (the banned list-in-cell encoding)
expect_raise("object-dtype values",
             lambda: PredictionFrame(np.array([[1.0], [2.0]], dtype=object), good_index))
# missing the explicit trailing sample axis (1-D)
expect_raise("1-D values (no sample axis)",
             lambda: PredictionFrame(np.array([1.0, 2.0], np.float32), good_index))
# rows disagree with the index length
expect_raise("rows != index length",
             lambda: PredictionFrame(np.zeros((3, 4), np.float32), good_index))
# float identifiers (integers cannot be NaN -> identifiers must be integer)
expect_raise("non-integer identifiers",
             lambda: SpatioTemporalIndex(np.array([0.5, 1.5]), np.array([1, 2], np.int64),
                                         SpatialLevel.PGM))

## 8 · Metadata & provenance

Every frame carries a typed `FrameMetadata` header — a frozen dataclass of all-optional,
validated fields (not a free-form dict, so consumers cannot diverge on key names). It is
the home for generic run identity (`run_id`, `data_version`); it travels through
`with_metadata` (zero-copy, §4) and survives serialization.

In [ ]:
meta = FrameMetadata(model="ensemble-A", run_id="run-2026-06-27", data_version="v2026.06.1")
print("to_dict (unset fields omitted):", meta.to_dict())
print("round-trips through a dict     :", FrameMetadata.from_dict(meta.to_dict()) == meta)

tagged = pred.with_metadata(meta)
with tempfile.TemporaryDirectory() as d:
    tagged.save(d)
    print("provenance survives save/load  :", PredictionFrame.load(d).metadata.run_id)

**Conformance coda.** The leaf *publishes* the contract checks consumers re-run in
their own CI (ADR-016), so every repo tests the same guarantee. `assert_frame_envelope`
is the shape/dtype/round-trip envelope any frame-like object must satisfy;
`assert_frame_contract` adds the spatiotemporal identifier rule. All three frames pass:

In [ ]:
from views_frames.conformance import assert_frame_contract, assert_frame_envelope

for name, frame in [("PredictionFrame", pred), ("TargetFrame", tgt), ("FeatureFrame", feat)]:
    assert_frame_envelope(frame)     # float32 + trailing axis + round-trip
    assert_frame_contract(frame)     # + integer (time, unit) of length n_rows
    print(f"{name:16s} satisfies the published frame contract ✓")

## The contract in one screen

| You saw | The guarantee |
|---|---|
| dense `float32 (N, S)` + integer ids (§1–2) | no list-in-cell `object` dtype; vectorisable, mmap-able |
| three sibling frames on one index (§3) | no base-class god-object; `cm`/`pgm` is a value, not a class |
| return-new, share-the-buffer (§4) | immutable value objects; zero-copy where safe, copy where needed |
| injected `(time, unit) → unit` remap (§5) | the leaf owns the *operation*; geography is never embedded (ADR-014) |
| npz + parquet round-trips (§6) | `frame in == frame out`; the `to_parquet` gap is a demand signal (D-11) |
| raises at construction (§7) | a malformed frame cannot exist; the guarantee is *structural* |
| typed `FrameMetadata` + published checks (§8) | provenance travels; every consumer re-runs the *same* contract |

Everything here is the **frozen** `views_frames` surface on synthetic data. Next:
[`02_summaries.ipynb`](02_summaries.ipynb) reads point estimates, intervals, exceedance
and tail risk *off the sample axis* of a `PredictionFrame` — and asks whether those
intervals are actually *honest*.